In [8]:
import yfinance as yf
import pandas as pd

stocks = [
"ADANIENT.NS","ADANIPORTS.NS","APOLLOHOSP.NS","ASIANPAINT.NS","AXISBANK.NS",
"BAJAJ-AUTO.NS","BAJAJFINSV.NS","BAJFINANCE.NS","BEL.NS","BHARTIARTL.NS",
"CIPLA.NS","COALINDIA.NS","DRREDDY.NS","EICHERMOT.NS","GRASIM.NS",
"HCLTECH.NS","HDFCBANK.NS","HDFCLIFE.NS","HINDALCO.NS","HINDUNILVR.NS",
"ICICIBANK.NS","INDIGO.NS","INFY.NS","ITC.NS","JIOFIN.NS","JSWSTEEL.NS",
"KOTAKBANK.NS","LT.NS","M&M.NS","MARUTI.NS","MAXHEALTH.NS","NESTLEIND.NS",
"NTPC.NS","ONGC.NS","POWERGRID.NS","RELIANCE.NS","SBILIFE.NS","SBIN.NS",
"SHRIRAMFIN.NS","SUNPHARMA.NS","TATACONSUM.NS","TATASTEEL.NS","TCS.NS",
"TECHM.NS","TITAN.NS","TMPV.NS","TRENT.NS","ULTRACEMCO.NS","WIPRO.NS","ETERNAL.NS"
]

all_data = []

for stock in stocks:

    df = yf.download(
        stock,
        start="2020-01-01",
        end="2025-12-31",
        progress=False
    )

    if df.empty:
        continue

    df.reset_index(inplace=True)

    # remove multiindex columns
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    df["Stock"] = stock

    
    # FIX missing Adj Close
    
    if "Adj Close" not in df.columns:
        df["Adj Close"] = df["Close"]

    df = df[["Date","Stock","Open","High","Low","Close","Adj Close","Volume"]]

    all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)

#final_df.to_excel("Indian_Nifty50_Stock_Data1.xlsx", index=False)

print("Excel file saved successfully")

Excel file saved successfully


In [16]:
final_df["Date"] = pd.to_datetime(final_df["Date"], dayfirst=True)

In [21]:
final_df = final_df.sort_values(["Stock","Date"])

In [22]:
final_df

,Date,Stock,Open,High,Low,Close,Adj Close,Volume
0,2020-01-01,ADANIENT.NS,207.025651,208.461950,204.697844,205.886520,205.886520,1553127
1,2020-01-02,ADANIENT.NS,206.035097,211.185971,205.539820,209.204865,209.204865,2991937
2,2020-01-03,ADANIENT.NS,208.263851,210.344020,203.855892,206.332275,206.332275,2512421
3,2020-01-06,ADANIENT.NS,205.787435,205.787435,195.881903,197.664902,197.664902,4353179
4,2020-01-07,ADANIENT.NS,198.655469,203.756813,198.655469,202.122406,202.122406,2966120
...,...,...,...,...,...,...,...,...
70309,2025-12-23,WIPRO.NS,265.154363,265.622293,263.409405,264.569458,264.569458,7572420
70310,2025-12-24,WIPRO.NS,264.121059,264.160061,260.475196,261.313538,261.313538,4694861
70311,2025-12-26,WIPRO.NS,260.475182,262.112893,259.051905,259.597809,259.597809,2899112
70312,2025-12-29,WIPRO.NS,259.295617,260.660406,256.965783,257.589661,257.589661,3755074


In [23]:
final_df.to_excel("Indian_Nifty50_Stock_Data.xlsx", index=False)

print("Excel file saved successfully")

Excel file saved successfully


In [10]:
nifty = yf.download("^NSEI", start="2020-01-01", end="2025-12-31")

nifty.reset_index(inplace=True)

# Flatten 
if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(0)

nifty = nifty[["Date", "Close"]]
nifty.rename(columns={"Close": "Index_Close"}, inplace=True)


[*********************100%***********************]  1 of 1 completed


In [12]:
nifty.duplicated().sum()

np.int64(0)

In [18]:
nifty["Date"] = pd.to_datetime(nifty["Date"], dayfirst=True)

In [19]:
nifty

Price,Date,Index_Close
0,2020-01-01,12182.500000
1,2020-01-02,12282.200195
2,2020-01-03,12226.650391
3,2020-01-06,11993.049805
4,2020-01-07,12052.950195
...,...,...
1480,2025-12-23,26177.150391
1481,2025-12-24,26142.099609
1482,2025-12-26,26042.300781
1483,2025-12-29,25942.099609


In [30]:
# Save to Excel
nifty.to_excel("nifty50.xlsx", index=False)

In [40]:
fundamental_data = []

for stock in stocks:
    ticker = yf.Ticker(stock)
    
    info = ticker.info
    financials = ticker.financials
    balance_sheet = ticker.balance_sheet

    try:
        net_income = financials.loc["Net Income"].iloc[0]
        total_equity = balance_sheet.loc["Stockholders Equity"].iloc[0]
        total_assets = balance_sheet.loc["Total Assets"].iloc[0]

        roe = net_income / total_equity if total_equity != 0 else None
        roa = net_income / total_assets if total_assets != 0 else None

    except Exception:
        roe = None
        roa = None

    fundamental_data.append({
        "Stock": stock,
        "Market_Cap": info.get("marketCap"),
        "PE_Ratio": info.get("trailingPE"),
        "PB_Ratio": info.get("priceToBook"),
        "ROE": roe,
        "ROA": roa,
        "Debt_to_Equity": info.get("debtToEquity"),
        "EPS": info.get("trailingEps"),
        "Revenue": info.get("totalRevenue"),
        "Profit_Margin": info.get("profitMargins"),
        "Dividend_Yield": info.get("dividendYield")
    })

fundamental_df = pd.DataFrame(fundamental_data)

In [41]:
print(fundamental_df.head())

           Stock     Market_Cap   PE_Ratio   PB_Ratio       ROE       ROA  \
0    ADANIENT.NS  2395368325120  15.893728   3.946950  0.141094  0.035829   
1  ADANIPORTS.NS  3024176545792  22.713272   4.227865  0.177661  0.081964   
2  APOLLOHOSP.NS  1066738384896  59.205173  11.730515  0.176065  0.069994   
3  ASIANPAINT.NS  2075548319744  53.900920  10.596065  0.189034  0.120746   
4    AXISBANK.NS  3609226379264  13.785613   1.741853  0.149839  0.016932   

   Debt_to_Equity     EPS       Revenue  Profit_Margin  Dividend_Yield  
0         177.767  110.66  949951594496        0.14111            0.07  
1          81.556   57.79  364866306048        0.34236            0.52  
2          83.606  125.31  242151997440        0.07441            0.26  
3          17.609   40.17  346495287296        0.11098            1.13  
4             NaN   84.24  768435683328        0.34170            0.08  


In [42]:
fundamental_df.fillna("NA", inplace=True)

C:\Users\anish\AppData\Local\Temp\ipykernel_1940\1950452583.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NA' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  fundamental_df.fillna("NA", inplace=True)


In [43]:
import yfinance as yf
import pandas as pd

data = []

for stock in stocks:
    print(f"Fetching {stock}...")
    
    ticker = yf.Ticker(stock)
    
    try:
        info = ticker.info
        
        sector = info.get("sector")
        industry = info.get("industry")
        
        # Handle missing values
        if not sector:
            sector = "Unknown"
        if not industry:
            industry = "Unknown"
    
    except:
        sector = "Unknown"
        industry = "Unknown"
    
    data.append({
        "Stock": stock,
        "Sector": sector,
        "Industry": industry
    })

sector_industry_df = pd.DataFrame(data)

print(sector_industry_df)

Fetching ADANIENT.NS...
Fetching ADANIPORTS.NS...
Fetching APOLLOHOSP.NS...
Fetching ASIANPAINT.NS...
Fetching AXISBANK.NS...
Fetching BAJAJ-AUTO.NS...
Fetching BAJAJFINSV.NS...
Fetching BAJFINANCE.NS...
Fetching BEL.NS...
Fetching BHARTIARTL.NS...
Fetching CIPLA.NS...
Fetching COALINDIA.NS...
Fetching DRREDDY.NS...
Fetching EICHERMOT.NS...
Fetching GRASIM.NS...
Fetching HCLTECH.NS...
Fetching HDFCBANK.NS...
Fetching HDFCLIFE.NS...
Fetching HINDALCO.NS...
Fetching HINDUNILVR.NS...
Fetching ICICIBANK.NS...
Fetching INDIGO.NS...
Fetching INFY.NS...
Fetching ITC.NS...
Fetching JIOFIN.NS...
Fetching JSWSTEEL.NS...
Fetching KOTAKBANK.NS...
Fetching LT.NS...
Fetching M&M.NS...
Fetching MARUTI.NS...
Fetching MAXHEALTH.NS...
Fetching NESTLEIND.NS...
Fetching NTPC.NS...
Fetching ONGC.NS...
Fetching POWERGRID.NS...
Fetching RELIANCE.NS...
Fetching SBILIFE.NS...
Fetching SBIN.NS...
Fetching SHRIRAMFIN.NS...
Fetching SUNPHARMA.NS...
Fetching TATACONSUM.NS...
Fetching TATASTEEL.NS...
Fetching TCS.N

In [44]:
fundamental_df = fundamental_df.merge(
    sector_industry_df,
    on="Stock",
    how="left"
)

In [23]:
fundamental_df.to_excel("nifty50_fundamental_data.xlsx", index=False)

### 1. Stock Return

In [25]:
final_df["stock_return"] = final_df.groupby("Stock")["Adj Close"].pct_change()

### 2. Index Return

In [27]:
nifty["index_return"] = nifty["Index_Close"].pct_change()
nifty["index_return"]

0            NaN
1       0.008184
2      -0.004523
3      -0.019106
4       0.004995
          ...   
1480    0.000181
1481   -0.001339
1482   -0.003818
1483   -0.003848
1484   -0.000125
Name: index_return, Length: 1485, dtype: float64

### 3. Alpha (OUTPERFORMANCE)

In [28]:
final_df = final_df.merge(nifty[["Date","index_return"]], on="Date", how="left")

final_df["alpha"] = final_df["stock_return"] - final_df["index_return"]


In [29]:
final_df["alpha"]

0             NaN
1        0.007933
2       -0.009208
3       -0.022901
4        0.017556
           ...   
71386   -0.004839
71387   -0.010968
71388   -0.002748
71389   -0.003888
71390   -0.002107
Name: alpha, Length: 71391, dtype: float64

### 4. Volatility (RISK)

In [31]:
final_df["volatility"] = (
    final_df.groupby("Stock")["stock_return"]
    .rolling(30)
    .std()
    .reset_index(0, drop=True)
)
final_df["volatility"]

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
           ...   
71386    0.010346
71387    0.010553
71388    0.010689
71389    0.010794
71390    0.010807
Name: volatility, Length: 71391, dtype: float64

### 5. Moving Averages (TREND SIGNAL)

In [32]:
final_df["MA50"] = final_df.groupby("Stock")["Adj Close"].transform(lambda x: x.rolling(50).mean())
final_df["MA200"] = final_df.groupby("Stock")["Adj Close"].transform(lambda x: x.rolling(200).mean())

final_df["MA50"]

final_df["MA200"]

0               NaN
1               NaN
2               NaN
3               NaN
4               NaN
            ...    
71386    243.873371
71387    243.841500
71388    243.776671
71389    243.697978
71390    243.621668
Name: MA200, Length: 71391, dtype: float64

### 6. Trend Signal (VERY POWERFUL for Q6)

In [33]:
final_df["trend_signal"] = final_df["MA50"] > final_df["MA200"]
final_df["trend_signal"]

0        False
1        False
2        False
3        False
4        False
         ...  
71386    False
71387    False
71388     True
71389     True
71390     True
Name: trend_signal, Length: 71391, dtype: bool

### 7. Sharpe Ratio (RISK-ADJUSTED RETURN)

In [34]:
final_df["sharpe"] = final_df["stock_return"] / final_df["volatility"]
final_df["sharpe"]

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
           ...   
71386   -0.450189
71387   -1.166187
71388   -0.614275
71389   -0.716683
71390   -0.206609
Name: sharpe, Length: 71391, dtype: float64

### 8. Downside Flag (for Q10)

In [36]:
final_df["market_crash"] = final_df["index_return"] < -0.02
final_df["market_crash"]

0        False
1        False
2        False
3        False
4        False
         ...  
71386    False
71387    False
71388    False
71389    False
71390    False
Name: market_crash, Length: 71391, dtype: bool

### 9. Sector Mapping (LIGHT JOIN — SAFE)

In [45]:
sector_map = fundamental_df[["Stock","Sector","Industry"]]

final_df = final_df.merge(sector_map, on="Stock", how="left")

### 10. Rolling Returns (MULTI-TIMEFRAME)

In [46]:
final_df["return_30d"] = final_df.groupby("Stock")["Adj Close"].pct_change(30)
final_df["return_90d"] = final_df.groupby("Stock")["Adj Close"].pct_change(90)
final_df["return_1y"] = final_df.groupby("Stock")["Adj Close"].pct_change(252)

### 11. Volatility Change (EARLY WARNING)

In [47]:
final_df["volatility_trend"] = final_df.groupby("Stock")["volatility"].pct_change()

C:\Users\anish\AppData\Local\Temp\ipykernel_1940\2243458460.py:1: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  final_df["volatility_trend"] = final_df.groupby("Stock")["volatility"].pct_change()


### 12. Price Momentum

In [49]:
final_df["momentum"] = final_df["Adj Close"] / final_df["MA50"]
final_df["momentum"]

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
           ...   
71386    1.087518
71387    1.072165
71388    1.063604
71389    1.054195
71390    1.051016
Name: momentum, Length: 71391, dtype: float64

In [51]:
final_df.fillna(0, inplace=True)

In [52]:
final_df.to_excel("final_stock_analysis.xlsx", index=False)

In [53]:
nifty.fillna(0, inplace=True)

In [54]:
nifty.to_excel("nifty50.xlsx", index=False)